# 03 - DNB Analysis (PRIMARY)

Explore Dynamic Network Biomarker results from `data/results/dnb/`.  
Reproduces Figures 1, 3-4: DNB score trajectories across disease stages,
network visualization of core proteins, GSEA enrichment, and sDNB distributions.  
**DNB is the primary analysis framework.** This notebook reads pre-computed results only.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy import stats

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

results_dir = os.path.join('..', config['paths']['results'], 'dnb')
print('DNB results directory:', results_dir)
print('Files available:', os.listdir(results_dir) if os.path.exists(results_dir) else 'DIRECTORY NOT FOUND')

## Figure 4 - DNB Score Across Disease Stages

DNB score should peak at the pre-transition stage (MCI pre-conversion),
not at established AD. The stage order reflects the disease timeline:
CN A-, CN A+, early MCI, late MCI, MCI converter (<12m), established AD.

In [ ]:
# Load DNB scores by stage
dnb_stages = pd.read_csv(os.path.join(results_dir, 'dnb_scores_by_stage.csv'))

stage_order = ['CN_amyloid_negative', 'CN_amyloid_positive', 'early_MCI', 'late_MCI',
               'MCI_converter_pre12m', 'AD']
stage_labels = ['CN A-', 'CN A+', 'Early MCI', 'Late MCI', 'MCI Conv\n(<12m)', 'AD']

# Filter and order stages
dnb_stages['stage'] = pd.Categorical(dnb_stages['stage'], categories=stage_order, ordered=True)
dnb_stages = dnb_stages.dropna(subset=['stage']).sort_values('stage')

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(dnb_stages)), dnb_stages['mean_dnb_score'],
       yerr=dnb_stages['sem_dnb_score'], color='darkorange', capsize=4, edgecolor='white')
ax.set_xticks(range(len(dnb_stages)))
ax.set_xticklabels(stage_labels[:len(dnb_stages)])
ax.set_ylabel('Mean DNB Score')
ax.set_title('Dynamic Network Biomarker Score Across Disease Stages')
plt.tight_layout()
plt.show()

print('DNB score by stage:')
print(dnb_stages[['stage', 'mean_dnb_score', 'sem_dnb_score', 'n']].to_string(index=False))

## Figure 5 - DNB Core Protein Network

Network visualization of proteins that consistently appear in the DNB group
across pre-conversion participants. Nodes colored by biological pathway from GSEA.

In [ ]:
# Load core proteins and GSEA results
core_proteins = pd.read_csv(os.path.join(results_dir, 'dnb_core_proteins.csv'))
gsea_path = os.path.join(results_dir, 'gsea_results')

print(f'Number of DNB core proteins: {len(core_proteins)}')
print(f'Top 10 by frequency:')
print(core_proteins.sort_values('frequency', ascending=False).head(10).to_string(index=False))

# Build a network from core proteins (edges = co-occurrence in DNB groups)
G = nx.Graph()
for _, row in core_proteins.iterrows():
    G.add_node(row['protein'], frequency=row['frequency'])

# Simple visualization -- edges added if pathway annotation exists
if 'pathway' in core_proteins.columns:
    pathway_groups = core_proteins.groupby('pathway')['protein'].apply(list)
    for pathway, proteins in pathway_groups.items():
        for i in range(len(proteins)):
            for j in range(i + 1, len(proteins)):
                G.add_edge(proteins[i], proteins[j])

fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.kamada_kawai_layout(G)
sizes = [G.nodes[n].get('frequency', 0.5) * 500 for n in G.nodes]
nx.draw(G, pos, ax=ax, node_size=sizes, node_color='darkorange',
        alpha=0.7, with_labels=True, font_size=6, edge_color='grey', width=0.5)
ax.set_title('DNB Core Protein Network')
plt.tight_layout()
plt.show()

## GSEA Pathway Enrichment

Top enriched pathways among DNB core proteins from fgsea analysis.

In [ ]:
# Load GSEA results if available
gsea_files = [f for f in os.listdir(gsea_path) if f.endswith('.csv')] if os.path.exists(gsea_path) else []

if gsea_files:
    gsea_df = pd.read_csv(os.path.join(gsea_path, gsea_files[0]))
    gsea_sig = gsea_df[gsea_df['padj'] < 0.25].sort_values('NES', ascending=False)

    print(f'Significant pathways (FDR < 0.25): {len(gsea_sig)}')

    if len(gsea_sig) > 0:
        fig, ax = plt.subplots(figsize=(8, max(4, len(gsea_sig) * 0.4)))
        ax.barh(range(len(gsea_sig)), gsea_sig['NES'], color='darkorange', edgecolor='white')
        ax.set_yticks(range(len(gsea_sig)))
        ax.set_yticklabels(gsea_sig['pathway'], fontsize=8)
        ax.set_xlabel('Normalized Enrichment Score (NES)')
        ax.set_title('GSEA: Enriched Pathways in DNB Core Proteins')
        plt.tight_layout()
        plt.show()
else:
    print('GSEA results not found. Run the R/gsea_analysis.R script first.')

## Figure 6 - sDNB Individual-Level Scores

Single-sample DNB scores for converter participants vs. time-to-conversion.  
Higher sDNB should correlate with proximity to conversion.

In [ ]:
# Load sDNB scores
sdnb = pd.read_csv(os.path.join(results_dir, 'sdnb_scores.csv'))

converters = sdnb[sdnb['TRAJECTORY'] == config['adni']['converter_group']].copy()
stable = sdnb[sdnb['TRAJECTORY'] == config['adni']['stable_group']].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: sDNB vs time-to-conversion
if 'MONTHS_TO_CONVERSION' in converters.columns:
    rho, pval = stats.spearmanr(converters['MONTHS_TO_CONVERSION'].dropna(),
                                 converters.loc[converters['MONTHS_TO_CONVERSION'].notna(), 'sdnb_score'])
    axes[0].scatter(converters['MONTHS_TO_CONVERSION'], converters['sdnb_score'],
                    c='firebrick', alpha=0.5, s=20)
    axes[0].set_xlabel('Months to Conversion')
    axes[0].set_ylabel('sDNB Score')
    axes[0].set_title(f'sDNB vs. Time to Conversion (rho={rho:.2f}, p={pval:.2e})')

# Violin: converters vs stable
parts = axes[1].violinplot([stable['sdnb_score'].dropna(), converters['sdnb_score'].dropna()],
                           positions=[0, 1], showmedians=True)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Stable MCI', 'MCI Converters'])
axes[1].set_ylabel('sDNB Score')
axes[1].set_title('sDNB Score by Group')

plt.tight_layout()
plt.show()